In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/College/FLEX-Med/flex-med

/content/drive/MyDrive/College/FLEX-Med/flex-med


In [ ]:
import os

# Define all the dataset paths you want to check
dataset_paths = [
    '/content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw',
    '/content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_balanced',
    '/content/drive/MyDrive/College/FLEX-Med/datasets/public_anchor',
    '/content/drive/MyDrive/College/FLEX-Med/datasets/public_test'
]

print(f"{'Dataset Name':<25} | {'ALL Count':<10} | {'HEM Count':<10} | {'Total':<10}")
print("-" * 65)

grand_total_all = 0
grand_total_hem = 0

for base_path in dataset_paths:
    # Get the folder name (e.g., 'client_allidb') for display
    dataset_name = os.path.basename(base_path)

    all_folder = os.path.join(base_path, 'all')
    hem_folder = os.path.join(base_path, 'hem')

    if os.path.exists(all_folder) and os.path.exists(hem_folder):
        # Count files, ignoring hidden files like .ipynb_checkpoints
        count_all = len([f for f in os.listdir(all_folder) if not f.startswith('.')])
        count_hem = len([f for f in os.listdir(hem_folder) if not f.startswith('.')])

        dataset_total = count_all + count_hem

        # Add to grand totals
        grand_total_all += count_all
        grand_total_hem += count_hem

        print(f"{dataset_name:<25} | {count_all:<10} | {count_hem:<10} | {dataset_total:<10}")
    else:
        print(f"{dataset_name:<25} | {'NOT FOUND':<10} | {'NOT FOUND':<10} | {'-'}")

print("-" * 65)
print(f"{'GRAND TOTAL':<25} | {grand_total_all:<10} | {grand_total_hem:<10} | {grand_total_all + grand_total_hem:<10}")

Dataset Name              | ALL Count  | HEM Count  | Total     
-----------------------------------------------------------------
all_idb2_raw              | 700        | 700        | 1400      
cnmc_balanced             | 2373       | 2373       | 4746      
public_anchor             | 658        | 658        | 1316      
public_test               | 658        | 658        | 1316      
-----------------------------------------------------------------
GRAND TOTAL               | 4389       | 4389       | 8778      


# **ALL_IDB2 Folder Preparation**


In [ ]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm

print("="*70)
print("STEP 1: ALL_IDB2 DATA PREPARATION")
print("="*70)

# --- CONFIGURATION ---
source_dir = '/content/drive/MyDrive/College/Datasets/ALL_IDB2/img'
base_dest_dir = '/content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw'

# Define sub-directories
dir_all = os.path.join(base_dest_dir, 'all')
dir_hem = os.path.join(base_dest_dir, 'hem')

# --- SETUP ---
os.makedirs(dir_all, exist_ok=True)
os.makedirs(dir_hem, exist_ok=True)

print(f"\nSource: {source_dir}")
print(f"Destination: {base_dest_dir}")
print("-" * 70)

# --- PROCESSING ---
files = [f for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]
count_all = 0
count_hem = 0
errors = 0

print("Organizing files by class label...")

for filename in tqdm(files, desc="Processing"):
    try:
        # Expected format: ImXXX_Y.tif where Y is the class label
        if '_' in filename:
            suffix = filename.split('_')[-1]
            label = suffix[0]  # Get the Y boolean (0 or 1)

            src_path = os.path.join(source_dir, filename)

            if label == '1':
                # Copy to 'all' (Leukemia/Blast cells)
                dest_path = os.path.join(dir_all, filename)
                shutil.copy2(src_path, dest_path)
                count_all += 1

            elif label == '0':
                # Copy to 'hem' (Healthy)
                dest_path = os.path.join(dir_hem, filename)
                shutil.copy2(src_path, dest_path)
                count_hem += 1
            else:
                print(f"Unknown label: {filename}")
                errors += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        errors += 1

# --- SUMMARY ---
print("\n" + "="*70)
print("ALL_IDB2 PREPARATION COMPLETE")
print("="*70)
print(f"ALL (Leukemia): {count_all}")
print(f"HEM (Healthy):  {count_hem}")
print(f"Total:          {count_all + count_hem}")
print(f"Balance Ratio:  {min(count_all, count_hem) / max(count_all, count_hem):.3f}")
if errors > 0:
    print(f"Errors:         {errors}")
print(f"\nLocation: {base_dest_dir}")

STEP 1: ALL_IDB2 DATA PREPARATION

Source: /content/drive/MyDrive/College/Datasets/ALL_IDB2/img
Destination: /content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw
----------------------------------------------------------------------
Organizing files by class label...


Processing: 100%|██████████| 260/260 [00:04<00:00, 59.18it/s]


ALL_IDB2 PREPARATION COMPLETE
ALL (Leukemia): 130
HEM (Healthy):  130
Total:          260
Balance Ratio:  1.000

Location: /content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw


# **ALL_IDB2 Data Augmentation**

In [ ]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import random
import torchvision.transforms as transforms

print("="*70)
print("STEP 1.5: ALL_IDB2 DATA AUGMENTATION")
print("="*70)

# --- CONFIGURATION ---
source_dir = '/content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw'
augmented_dir = '/content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_augmented'

# Target: 1000 images per class (balanced)
TARGET_IMAGES_PER_CLASS = 1000

random.seed(42)

print(f"\nSource: {source_dir}")
print(f"Destination: {augmented_dir}")
print(f"Target per class: {TARGET_IMAGES_PER_CLASS} images")
print("-" * 70)

# --- SETUP OUTPUT DIRECTORIES ---
for class_name in ['all', 'hem']:
    os.makedirs(os.path.join(augmented_dir, class_name), exist_ok=True)

# --- DEFINE AUGMENTATION TRANSFORMS ---
# Multiple augmentation strategies for diversity
augmentation_transforms = {
    'original': transforms.Compose([
        transforms.Resize((128, 128)),
    ]),

    'flip_h': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=1.0),
    ]),

    'flip_v': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomVerticalFlip(p=1.0),
    ]),

    'rotate_15': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomRotation(degrees=(15, 15)),
    ]),

    'rotate_30': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomRotation(degrees=(30, 30)),
    ]),

    'rotate_neg15': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomRotation(degrees=(-15, -15)),
    ]),

    'color_jitter': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    ]),

    'gaussian_blur': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    ]),

    'affine': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    ]),

    'perspective': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomPerspective(distortion_scale=0.2, p=1.0),
    ]),

    'combo_1': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomRotation(degrees=15),
    ]),

    'combo_2': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
    ]),

    'combo_3': transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    ]),
}

# --- AUGMENTATION FUNCTION ---
def augment_class(class_name, source_class_dir, output_class_dir, target_count):
    """
    Augment images in a class to reach target count.
    """
    print(f"\n[{class_name.upper()}] Processing...")

    # Get original images
    original_images = [
        f for f in os.listdir(source_class_dir)
        if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp', '.tif', '.tiff'))
    ]

    num_original = len(original_images)
    print(f"  Original images: {num_original}")
    print(f"  Target images: {target_count}")
    print(f"  Need to generate: {target_count - num_original} augmented images")

    if num_original >= target_count:
        print(f"  Already have enough images! Copying {target_count} images...")
        # Just copy the original images
        for i, img_file in enumerate(tqdm(original_images[:target_count], desc=f"  Copying {class_name}")):
            src = os.path.join(source_class_dir, img_file)
            dst = os.path.join(output_class_dir, img_file)
            shutil.copy2(src, dst)
        return target_count

    # Copy all original images first
    print(f"  Copying original images...")
    for img_file in tqdm(original_images, desc=f"  Original {class_name}", leave=False):
        src = os.path.join(source_class_dir, img_file)
        dst = os.path.join(output_class_dir, f"orig_{img_file}")
        shutil.copy2(src, dst)

    # Calculate how many augmented versions we need
    num_to_generate = target_count - num_original
    augmentations_per_image = num_to_generate // num_original + 1

    print(f"  Generating ~{augmentations_per_image} augmented versions per image...")

    # Generate augmented images
    generated_count = 0
    transform_names = list(augmentation_transforms.keys())

    for img_file in tqdm(original_images, desc=f"  Augmenting {class_name}"):
        if generated_count >= num_to_generate:
            break

        # Load original image
        img_path = os.path.join(source_class_dir, img_file)
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"    Error loading {img_file}: {e}")
            continue

        # Apply different augmentations
        base_name = os.path.splitext(img_file)[0]
        ext = os.path.splitext(img_file)[1]

        # Shuffle transforms for variety
        shuffled_transforms = transform_names.copy()
        random.shuffle(shuffled_transforms)

        for i, transform_name in enumerate(shuffled_transforms):
            if generated_count >= num_to_generate:
                break

            if transform_name == 'original':
                continue  # Skip original (already copied)

            try:
                # Apply augmentation
                transform = augmentation_transforms[transform_name]
                augmented_img = transform(img)

                # Save augmented image
                aug_filename = f"aug_{transform_name}_{base_name}{ext}"
                aug_path = os.path.join(output_class_dir, aug_filename)
                augmented_img.save(aug_path)

                generated_count += 1

            except Exception as e:
                print(f"    Error augmenting {img_file} with {transform_name}: {e}")
                continue

    total_images = num_original + generated_count
    print(f"  ✓ Complete: {num_original} original + {generated_count} augmented = {total_images} total")

    return total_images

# --- PROCESS BOTH CLASSES ---
total_stats = {}

for class_name in ['all', 'hem']:
    source_class_dir = os.path.join(source_dir, class_name)
    output_class_dir = os.path.join(augmented_dir, class_name)

    if not os.path.exists(source_class_dir):
        print(f"[ERROR] {source_class_dir} not found!")
        continue

    total = augment_class(
        class_name=class_name,
        source_class_dir=source_class_dir,
        output_class_dir=output_class_dir,
        target_count=TARGET_IMAGES_PER_CLASS
    )

    total_stats[class_name] = total

# --- FINAL SUMMARY ---
print("\n" + "="*70)
print("AUGMENTATION COMPLETE")
print("="*70)

for class_name in ['all', 'hem']:
    if class_name in total_stats:
        count = total_stats[class_name]
        print(f"\n{class_name.upper()} (Leukemia)" if class_name == 'all' else f"\n{class_name.upper()} (Healthy):")
        print(f"  Total images: {count}")
        print(f"  Target: {TARGET_IMAGES_PER_CLASS}")
        print(f"  Status: {'✓ Target reached' if count >= TARGET_IMAGES_PER_CLASS else '⚠ Below target'}")

total_all = total_stats.get('all', 0)
total_hem = total_stats.get('hem', 0)
grand_total = total_all + total_hem

print(f"\n{'='*70}")
print(f"GRAND TOTAL: {grand_total} images")
print(f"  ALL: {total_all}")
print(f"  HEM: {total_hem}")
print(f"  Balance: {min(total_all, total_hem) / max(total_all, total_hem):.3f}")

print(f"\n✓ Augmented dataset location: {augmented_dir}")
print("\nNext step: Use 'all_idb2_augmented' instead of 'all_idb2_raw' in your FL config")
print("="*70)

STEP 1.5: ALL_IDB2 DATA AUGMENTATION

Source: /content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw
Destination: /content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_augmented
Target per class: 1000 images
----------------------------------------------------------------------

[ALL] Processing...
  Original images: 130
  Target images: 1000
  Need to generate: 870 augmented images
  Copying original images...


  Generating ~7 augmented versions per image...


  Augmenting all:  56%|█████▌    | 73/130 [00:13<00:10,  5.27it/s]


  ✓ Complete: 130 original + 870 augmented = 1000 total

[HEM] Processing...
  Original images: 130
  Target images: 1000
  Need to generate: 870 augmented images
  Copying original images...


  Generating ~7 augmented versions per image...


  Augmenting hem:  56%|█████▌    | 73/130 [00:12<00:09,  5.88it/s]

  ✓ Complete: 130 original + 870 augmented = 1000 total

AUGMENTATION COMPLETE

ALL (Leukemia)
  Total images: 1000
  Target: 1000
  Status: ✓ Target reached

HEM (Healthy):
  Total images: 1000
  Target: 1000
  Status: ✓ Target reached

GRAND TOTAL: 2000 images
  ALL: 1000
  HEM: 1000
  Balance: 1.000

✓ Augmented dataset location: /content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_augmented

Next step: Use 'all_idb2_augmented' instead of 'all_idb2_raw' in your FL config


# **CNMC Folder Preparation**

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

print("="*70)
print("STEP 2: CNMC DATA PREPARATION + BALANCING")
print("="*70)

# --- CONFIGURATION ---
source_root = '/content/drive/MyDrive/College/Datasets/CNMC/training_data'
raw_dest_root = '/content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_raw'
balanced_dest_root = '/content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_balanced'

folds = ['fold_0', 'fold_1', 'fold_2']
categories = ['all', 'hem']

random.seed(42)

# --- STEP 2A: MERGE FOLDS ---
print("\n[2A] Merging folds into single dataset...")

for cat in categories:
    os.makedirs(os.path.join(raw_dest_root, cat), exist_ok=True)

total_copied = 0
errors = 0

for fold in folds:
    print(f"\nProcessing {fold}...")

    for cat in categories:
        src_dir = os.path.join(source_root, fold, cat)
        dest_dir = os.path.join(raw_dest_root, cat)

        if not os.path.exists(src_dir):
            print(f"  [WARNING] Not found: {src_dir}")
            continue

        files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]

        for filename in tqdm(files, desc=f"  {cat}", leave=False):
            try:
                src_file = os.path.join(src_dir, filename)
                # Prepend fold name to prevent duplicates
                new_filename = f"{fold}_{filename}"
                dest_file = os.path.join(dest_dir, new_filename)

                shutil.copy2(src_file, dest_file)
                total_copied += 1

            except Exception as e:
                print(f"  Error: {filename}: {e}")
                errors += 1

# Count merged files
count_all_raw = len(os.listdir(os.path.join(raw_dest_root, 'all')))
count_hem_raw = len(os.listdir(os.path.join(raw_dest_root, 'hem')))

print("\n" + "-"*70)
print("MERGE COMPLETE")
print("-"*70)
print(f"ALL (Leukemia): {count_all_raw}")
print(f"HEM (Healthy):  {count_hem_raw}")
print(f"Total:          {count_all_raw + count_hem_raw}")
print(f"Imbalance Ratio: {count_all_raw / count_hem_raw:.2f}:1 (ALL:HEM)")
print(f"Location: {raw_dest_root}")

# --- STEP 2B: BALANCE CLASSES ---
print("\n" + "="*70)
print("[2B] Balancing CNMC (Undersampling ALL to match HEM)")
print("="*70)

for cat in categories:
    os.makedirs(os.path.join(balanced_dest_root, cat), exist_ok=True)

# Get file lists
all_path = os.path.join(raw_dest_root, 'all')
hem_path = os.path.join(raw_dest_root, 'hem')

all_files = [f for f in os.listdir(all_path) if os.path.isfile(os.path.join(all_path, f))]
hem_files = [f for f in os.listdir(hem_path) if os.path.isfile(os.path.join(hem_path, f))]

n_all = len(all_files)
n_hem = len(hem_files)

# Determine target (match minority class)
target_count = min(n_all, n_hem)

print(f"\nOriginal counts:")
print(f"  ALL: {n_all}")
print(f"  HEM: {n_hem}")
print(f"  Minority class: {target_count}")
print(f"\nTarget balanced count: {target_count} per class")

# Undersample ALL
random.shuffle(all_files)
sampled_all = all_files[:target_count]

print(f"\nCopying {target_count} ALL images (undersampled)...")
for filename in tqdm(sampled_all, desc="ALL"):
    src = os.path.join(all_path, filename)
    dst = os.path.join(balanced_dest_root, 'all', filename)
    shutil.copy2(src, dst)

# Copy all HEM
print(f"Copying {n_hem} HEM images (unchanged)...")
for filename in tqdm(hem_files, desc="HEM"):
    src = os.path.join(hem_path, filename)
    dst = os.path.join(balanced_dest_root, 'hem', filename)
    shutil.copy2(src, dst)

# Verify
final_all = len(os.listdir(os.path.join(balanced_dest_root, 'all')))
final_hem = len(os.listdir(os.path.join(balanced_dest_root, 'hem')))

print("\n" + "="*70)
print("CNMC BALANCING COMPLETE")
print("="*70)
print(f"ALL (Leukemia): {final_all}")
print(f"HEM (Healthy):  {final_hem}")
print(f"Total:          {final_all + final_hem}")
print(f"Balance Ratio:  {min(final_all, final_hem) / max(final_all, final_hem):.3f} (1.0 = perfect)")
print(f"\nLocation: {balanced_dest_root}")

STEP 2: CNMC DATA PREPARATION + BALANCING

[2A] Merging folds into single dataset...

Processing fold_0...



Processing fold_1...



Processing fold_2...



----------------------------------------------------------------------
MERGE COMPLETE
----------------------------------------------------------------------
ALL (Leukemia): 7273
HEM (Healthy):  3389
Total:          10662
Imbalance Ratio: 2.15:1 (ALL:HEM)
Location: /content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_raw

[2B] Balancing CNMC (Undersampling ALL to match HEM)

Original counts:
  ALL: 7273
  HEM: 3389
  Minority class: 3389

Target balanced count: 3389 per class

Copying 3389 ALL images (undersampled)...


ALL: 100%|██████████| 3389/3389 [02:38<00:00, 21.33it/s]


Copying 3389 HEM images (unchanged)...


HEM: 100%|██████████| 3389/3389 [01:23<00:00, 40.43it/s]



CNMC BALANCING COMPLETE
ALL (Leukemia): 3389
HEM (Healthy):  3389
Total:          6778
Balance Ratio:  1.000 (1.0 = perfect)

Location: /content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_balanced


In [ ]:
import os
import shutil
import random
from tqdm import tqdm

print("="*70)
print("CNMC BALANCING (cnmc_raw → cnmc_balanced)")
print("="*70)

# --- CONFIGURATION ---
raw_dest_root = '/content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_raw'
balanced_dest_root = '/content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_balanced'

categories = ['all', 'hem']
random.seed(42)

# --- SETUP OUTPUT DIRECTORIES ---
for cat in categories:
    os.makedirs(os.path.join(balanced_dest_root, cat), exist_ok=True)

# --- GET FILE LISTS ---
all_path = os.path.join(raw_dest_root, 'all')
hem_path = os.path.join(raw_dest_root, 'hem')

all_files = [f for f in os.listdir(all_path) if os.path.isfile(os.path.join(all_path, f))]
hem_files = [f for f in os.listdir(hem_path) if os.path.isfile(os.path.join(hem_path, f))]

n_all = len(all_files)
n_hem = len(hem_files)

# Determine target (match minority class)
target_count = min(n_all, n_hem)

print(f"\nOriginal counts:")
print(f"  ALL: {n_all}")
print(f"  HEM: {n_hem}")
print(f"  Minority class: {target_count}")
print(f"\nTarget balanced count: {target_count} per class")

# --- UNDERSAMPLE ALL ---
random.shuffle(all_files)
sampled_all = all_files[:target_count]

print(f"\nCopying {target_count} ALL images (undersampled)...")
for filename in tqdm(sampled_all, desc="ALL"):
    src = os.path.join(all_path, filename)
    dst = os.path.join(balanced_dest_root, 'all', filename)
    shutil.copy2(src, dst)

# --- COPY ALL HEM ---
print(f"Copying {n_hem} HEM images (unchanged)...")
for filename in tqdm(hem_files, desc="HEM"):
    src = os.path.join(hem_path, filename)
    dst = os.path.join(balanced_dest_root, 'hem', filename)
    shutil.copy2(src, dst)

# --- VERIFY ---
final_all = len(os.listdir(os.path.join(balanced_dest_root, 'all')))
final_hem = len(os.listdir(os.path.join(balanced_dest_root, 'hem')))

print("\n" + "="*70)
print("CNMC BALANCING COMPLETE")
print("="*70)
print(f"ALL (Leukemia): {final_all}")
print(f"HEM (Healthy):  {final_hem}")
print(f"Total:          {final_all + final_hem}")
print(f"Balance Ratio:  {min(final_all, final_hem) / max(final_all, final_hem):.3f} (1.0 = perfect)")
print(f"\nLocation: {balanced_dest_root}")


# **Public Anchor and Test Dataset Preparation**



In [ ]:
import os
import shutil
import random
from tqdm import tqdm
from collections import defaultdict

print("="*70)
print("STEP 3: CREATE PUBLIC ANCHOR & PUBLIC TEST DATASETS")
print("="*70)

# --- CONFIGURATION ---
datasets = {
    'cnmc': '/content/drive/MyDrive/College/FLEX-Med/datasets/cnmc_balanced',
    'allidb2': '/content/drive/MyDrive/College/FLEX-Med/datasets/all_idb2_raw'
}

# Public datasets will be at the same level as source datasets
base_dir = '/content/drive/MyDrive/College/FLEX-Med/datasets'
public_anchor_path = os.path.join(base_dir, 'public_anchor')
public_test_path = os.path.join(base_dir, 'public_test')

random.seed(42)

print(f"\nConfiguration:")
print(f"  Strategy: Equal contribution from each dataset (Option 1)")
print(f"  Method: 20% anchor + 10% test of smallest dataset per class")
print("-"*70)

# --- STEP 1: FIND MINIMUM DATASET SIZE ---
print("\n[STEP 1] Scanning datasets to find minimum size...")

dataset_sizes = {}
for dataset_name, dataset_path in datasets.items():
    all_path = os.path.join(dataset_path, 'all')
    hem_path = os.path.join(dataset_path, 'hem')
    
    if not os.path.exists(all_path) or not os.path.exists(hem_path):
        print(f"  [ERROR] Dataset not found: {dataset_path}")
        continue
    
    # Count images per class
    all_count = len([f for f in os.listdir(all_path) 
                     if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp', '.tif', '.tiff'))])
    hem_count = len([f for f in os.listdir(hem_path) 
                     if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp', '.tif', '.tiff'))])
    
    # Use minimum of both classes to ensure balance
    min_class_size = min(all_count, hem_count)
    dataset_sizes[dataset_name] = {
        'all': all_count,
        'hem': hem_count,
        'min': min_class_size
    }
    
    print(f"  {dataset_name}: ALL={all_count}, HEM={hem_count}, Min={min_class_size}")

# Find the overall minimum across all datasets
min_dataset_size = min(size['min'] for size in dataset_sizes.values())
print(f"\n  → Smallest dataset size per class: {min_dataset_size}")

# --- STEP 2: CALCULATE TARGET SAMPLES ---
PUBLIC_ANCHOR_RATIO = 0.20  # 20% for public anchor
PUBLIC_TEST_RATIO = 0.10    # 10% for public test

samples_per_class_per_dataset_anchor = int(min_dataset_size * PUBLIC_ANCHOR_RATIO)
samples_per_class_per_dataset_test = int(min_dataset_size * PUBLIC_TEST_RATIO)

print(f"\n[STEP 2] Calculating target samples per dataset:")
print(f"  Public Anchor: {samples_per_class_per_dataset_anchor} samples/class/dataset ({PUBLIC_ANCHOR_RATIO*100:.0f}% of {min_dataset_size})")
print(f"  Public Test:   {samples_per_class_per_dataset_test} samples/class/dataset ({PUBLIC_TEST_RATIO*100:.0f}% of {min_dataset_size})")
print(f"\n  Expected Public Anchor Total:")
print(f"    - {samples_per_class_per_dataset_anchor} × {len(datasets)} datasets × 2 classes = {samples_per_class_per_dataset_anchor * len(datasets) * 2} images")
print(f"    - {samples_per_class_per_dataset_anchor * len(datasets)} ALL + {samples_per_class_per_dataset_anchor * len(datasets)} HEM")
print(f"\n  Expected Public Test Total:")
print(f"    - {samples_per_class_per_dataset_test} × {len(datasets)} datasets × 2 classes = {samples_per_class_per_dataset_test * len(datasets) * 2} images")
print(f"    - {samples_per_class_per_dataset_test * len(datasets)} ALL + {samples_per_class_per_dataset_test * len(datasets)} HEM")
print("-"*70)

# --- CREATE OUTPUT DIRECTORIES ---
for class_name in ['all', 'hem']:
    os.makedirs(os.path.join(public_anchor_path, class_name), exist_ok=True)
    os.makedirs(os.path.join(public_test_path, class_name), exist_ok=True)

print(f"\nOutput directories created:")
print(f"  - {public_anchor_path}")
print(f"  - {public_test_path}")
print("-"*70)

# --- STEP 3: EXTRACT FIXED SAMPLES FROM EACH DATASET ---
def extract_equal_samples(source_path, dataset_name, anchor_samples, test_samples):
    """
    Extract exactly anchor_samples and test_samples per class from the dataset.
    This ensures equal contribution from all datasets regardless of their size.
    """
    print(f"\n{'='*70}")
    print(f"Processing: {dataset_name.upper()}")
    print(f"Source: {source_path}")
    print(f"Target: {anchor_samples} anchor + {test_samples} test per class")
    print('='*70)

    stats = defaultdict(lambda: defaultdict(int))

    for class_name in ['all', 'hem']:
        class_path = os.path.join(source_path, class_name)

        if not os.path.exists(class_path):
            print(f"  [ERROR] Class folder not found: {class_path}")
            continue

        # Get all images
        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp', '.tif', '.tiff'))
        ]

        if not images:
            print(f"  [WARNING] No images found in {class_path}")
            continue

        n_total = len(images)
        
        # Check if we have enough images
        needed = anchor_samples + test_samples
        if n_total < needed:
            print(f"  [WARNING] {class_name}: Only {n_total} images available, need {needed}")
            anchor_samples_actual = int(n_total * PUBLIC_ANCHOR_RATIO / (PUBLIC_ANCHOR_RATIO + PUBLIC_TEST_RATIO))
            test_samples_actual = n_total - anchor_samples_actual
        else:
            anchor_samples_actual = anchor_samples
            test_samples_actual = test_samples

        # Shuffle for random selection
        random.shuffle(images)

        # Select images
        anchor_imgs = images[:anchor_samples_actual]
        test_imgs = images[anchor_samples_actual:anchor_samples_actual + test_samples_actual]
        private_imgs = images[anchor_samples_actual + test_samples_actual:]

        # MOVE to public_anchor
        print(f"\n[{class_name.upper()}] Moving {len(anchor_imgs)} to public_anchor/{class_name}/...")
        for img in tqdm(anchor_imgs, desc=f"  Public Anchor", leave=False):
            src = os.path.join(class_path, img)
            # Prefix with dataset name to prevent filename collisions
            dst_filename = f"{dataset_name}_{img}"
            dst = os.path.join(public_anchor_path, class_name, dst_filename)
            shutil.move(src, dst)
            stats['public_anchor'][class_name] += 1

        # MOVE to public_test
        print(f"[{class_name.upper()}] Moving {len(test_imgs)} to public_test/{class_name}/...")
        for img in tqdm(test_imgs, desc=f"  Public Test", leave=False):
            src = os.path.join(class_path, img)
            # Prefix with dataset name to prevent filename collisions
            dst_filename = f"{dataset_name}_{img}"
            dst = os.path.join(public_test_path, class_name, dst_filename)
            shutil.move(src, dst)
            stats['public_test'][class_name] += 1

        # Count what remains (private data)
        stats[f'private_{dataset_name}'][class_name] = len(private_imgs)

        # Print class summary
        print(f"\n[{class_name.upper()}] Split Summary:")
        print(f"  Total Original:       {n_total}")
        print(f"  → Public Anchor:      {len(anchor_imgs)} ({len(anchor_imgs)/n_total*100:.1f}%)")
        print(f"  → Public Test:        {len(test_imgs)} ({len(test_imgs)/n_total*100:.1f}%)")
        print(f"  → Private (remains):  {len(private_imgs)} ({len(private_imgs)/n_total*100:.1f}%)")

    return stats

# --- PROCESS ALL DATASETS WITH EQUAL SAMPLING ---
all_stats = defaultdict(lambda: defaultdict(int))

for dataset_name, dataset_path in datasets.items():
    dataset_stats = extract_equal_samples(
        source_path=dataset_path,
        dataset_name=dataset_name,
        anchor_samples=samples_per_class_per_dataset_anchor,
        test_samples=samples_per_class_per_dataset_test
    )
    
    # Aggregate stats
    for split_name, class_stats in dataset_stats.items():
        for class_name, count in class_stats.items():
            all_stats[split_name][class_name] += count

# --- FINAL SUMMARY ---
print("\n" + "="*70)
print("DATA SPLIT SUMMARY - EQUAL CONTRIBUTION APPROACH")
print("="*70)

# Public datasets (combined)
for split_name in ['public_anchor', 'public_test']:
    all_count = all_stats[split_name]['all']
    hem_count = all_stats[split_name]['hem']
    total = all_count + hem_count

    if total > 0:
        balance = min(all_count, hem_count) / max(all_count, hem_count) if max(all_count, hem_count) > 0 else 1.0

        print(f"\n{split_name.upper().replace('_', ' ')} (Equal contribution from all datasets):")
        print(f"  ALL (Leukemia): {all_count:>5} ({len(datasets)} datasets × {all_count//len(datasets)} samples)")
        print(f"  HEM (Healthy):  {hem_count:>5} ({len(datasets)} datasets × {hem_count//len(datasets)} samples)")
        print(f"  Total:          {total:>5}")
        print(f"  Balance:        {balance:.3f} (1.0 = perfect)")
        print(f"  Dataset Bias:   NONE (Equal {all_count//len(datasets)} samples per dataset)")

# Private datasets (remaining in source folders)
print("\n" + "-"*70)
print("PRIVATE DATA (Remaining in source folders):")
print("-"*70)

for split_name in ['private_cnmc', 'private_allidb2']:
    all_count = all_stats[split_name]['all']
    hem_count = all_stats[split_name]['hem']
    total = all_count + hem_count

    if total > 0:
        balance = min(all_count, hem_count) / max(all_count, hem_count) if max(all_count, hem_count) > 0 else 1.0
        dataset_name = split_name.replace('private_', '')
        
        # Calculate percentage remaining
        original_size = dataset_sizes.get(dataset_name, {}).get('min', 1)
        retention_pct = (total / (original_size * 2)) * 100 if original_size > 0 else 0

        print(f"\n{split_name.upper().replace('_', ' ')}:")
        print(f"  ALL (Leukemia): {all_count:>5}")
        print(f"  HEM (Healthy):  {hem_count:>5}")
        print(f"  Total:          {total:>5} ({retention_pct:.1f}% of original retained)")
        print(f"  Balance:        {balance:.3f}")

# --- VERIFY STRUCTURE AND CONTRIBUTION ---
print("\n" + "="*70)
print("VERIFICATION: EQUAL CONTRIBUTION & CLASS BALANCE")
print("="*70)

for name, path in [('public_anchor', public_anchor_path), ('public_test', public_test_path)]:
    all_dir = os.path.join(path, 'all')
    hem_dir = os.path.join(path, 'hem')
    
    if os.path.exists(all_dir) and os.path.exists(hem_dir):
        all_files = os.listdir(all_dir)
        hem_files = os.listdir(hem_dir)
        
        all_count = len(all_files)
        hem_count = len(hem_files)
        
        # Count contribution per dataset
        contribution = defaultdict(lambda: {'all': 0, 'hem': 0})
        for f in all_files:
            dataset = f.split('_')[0]
            contribution[dataset]['all'] += 1
        for f in hem_files:
            dataset = f.split('_')[0]
            contribution[dataset]['hem'] += 1
        
        print(f"\n✓ {name}: LABELED & BALANCED")
        print(f"  Total: {all_count} ALL + {hem_count} HEM = {all_count + hem_count}")
        print(f"  Class Balance: {min(all_count, hem_count) / max(all_count, hem_count):.3f}")
        print(f"\n  Dataset Contribution:")
        for ds_name in sorted(contribution.keys()):
            ds_all = contribution[ds_name]['all']
            ds_hem = contribution[ds_name]['hem']
            ds_total = ds_all + ds_hem
            print(f"    {ds_name}: {ds_all} ALL + {ds_hem} HEM = {ds_total} ({ds_total/(all_count+hem_count)*100:.1f}%)")
    else:
        print(f"✗ {name}: MISSING CLASS FOLDERS!")

print("\n" + "="*70)
print("✓ DATA PREPARATION COMPLETE - EQUAL CONTRIBUTION ACHIEVED")
print("="*70)

print(f"\nFinal Directory Structure:")
print(f"/datasets/")
print(f"  ├── cnmc_balanced/          (private data remains here)")
print(f"  │   ├── all/")
print(f"  │   └── hem/")
print(f"  ├── all_idb2_raw/           (private data remains here)")
print(f"  │   ├── all/")
print(f"  │   └── hem/")
print(f"  ├── public_anchor/          (EQUAL contribution, LABELED, combined)")
print(f"  │   ├── all/")
print(f"  │   └── hem/")
print(f"  └── public_test/            (EQUAL contribution, LABELED, combined)")
print(f"      ├── all/")
print(f"      └── hem/")

print(f"\nKey Improvements:")
print(f"  ✓ Equal contribution from each dataset (no bias)")
print(f"  ✓ Perfect class balance (ALL = HEM)")
print(f"  ✓ Based on smallest dataset to avoid imbalance")
print(f"  ✓ 20% anchor + 10% test sampling ratio from smallest dataset")

print(f"\nNext steps:")
print(f"  1. Update task.py:")
print(f"     DATA_ROOT = '{base_dir}'")
print(f"     PUBLIC_PATH = '{public_anchor_path}'")
print(f"     GLOBAL_TEST_PATH = '{public_test_path}'")
print(f"\n  2. Update data.json:")
print(f"     Client 0: dataset_path = '{datasets['cnmc']}'")
print(f"     Client 1: dataset_path = '{datasets['allidb2']}'")
print(f"\n  3. Run: flwr run .")